# Exploratory analysis with Opteryx Core

Query a Parquet dataset with SQL inside a notebook, take a fast first look at it, chart a summary with pandas, and iterate rows in Python.

This notebook accompanies [Exploratory Analysis in Notebooks](https://docs.opteryx.app/docs/guides/exploratory-analysis) on docs.opteryx.app. It runs in Opteryx Core, in the notebook's own process. No account needed.

In [ ]:
%pip install -q opteryx-core pandas pyarrow matplotlib

## Get the data

The guide queries a `transactions` dataset you won't have, so this notebook uses `space_missions.parquet` instead: 4,630 orbital launch attempts since Sputnik. Its columns stand in for the guide's:

| guide | here |
|---|---|
| `data.transactions` | `data.missions` |
| `category` | `Company` |
| `amount` | `Price` (launch cost, millions of USD; often missing) |
| `ordered_at` | `Lauched_at` (spelled that way in the file) |
| `status` | `Mission_Status` |

A dataset is a folder, and `data.missions` reads from `./data/missions/`, so the file goes there.

In [ ]:
import os
import urllib.request

os.makedirs("data/missions", exist_ok=True)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/mabel-dev/opteryx-core/main/testdata/missions/space_missions.parquet",
    "data/missions/space_missions.parquet",
)

## Setup

Register the `data` workspace with the `DiskConnector` and open a session.

In [ ]:
import opteryx
from opteryx.connectors import DiskConnector

opteryx.register_workspace("data", DiskConnector)
session = opteryx.session()

## Querying without loading everything

Aggregate in SQL and only pull back the summary. The scan, filter and grouping run inside Opteryx; only one row per company reaches Python.

Compare a timestamp column with a `CAST(... AS TIMESTAMP)`, not a bare string.

In [ ]:
for morsel in session.execute_to_morsels("""
    SELECT
        Company,
        COUNT(*)    AS n,
        AVG(Price)  AS avg_price
      FROM data.missions
     WHERE Lauched_at >= CAST('2000-01-01' AS TIMESTAMP)
     GROUP BY Company
     ORDER BY n DESC
"""):
    print(morsel.to_arrow())

## A fast first look

A sample of rows:

In [ ]:
for morsel in session.execute_to_morsels("SELECT * FROM data.missions LIMIT 20"):
    print(morsel.to_arrow())

The row count. For a small result, concatenate the morsels into one Arrow table and show it as pandas:

In [ ]:
import pyarrow

morsels = session.execute_to_morsels("SELECT COUNT(*) FROM data.missions")
pyarrow.concat_tables(m.to_arrow() for m in morsels).to_pandas()

Distinct values and their frequencies:

In [ ]:
morsels = session.execute_to_morsels("""
    SELECT Mission_Status, COUNT(*) AS n
      FROM data.missions
     GROUP BY Mission_Status
     ORDER BY n DESC
""")
pyarrow.concat_tables(m.to_arrow() for m in morsels).to_pandas()

## Straight to a chart

Group in SQL, convert the small result to pandas, and plot it. This groups by year rather than the guide's month, since launch prices are sparse before 2000.

In [ ]:
morsels = session.execute_to_morsels("""
    SELECT
        FORMAT_TIMESTAMP('%Y', Lauched_at) AS year,
        SUM(Price)                         AS spend
      FROM data.missions
     WHERE Lauched_at >= CAST('2000-01-01' AS TIMESTAMP)
     GROUP BY year
     ORDER BY year
""")
yearly = pyarrow.concat_tables(morsel.to_arrow() for morsel in morsels).to_pandas()

yearly.plot(x="year", y="spend")

## Working row by row

Iterate a morsel directly when you need Python logic per row. Each row is a named tuple.

In [ ]:
for morsel in session.execute_to_morsels("SELECT name, mass, gravity FROM $planets"):
    for row in morsel:
        print(row.name, row.mass, row.gravity)

## Try it against sample data

`$planets` is built in and touches no workspace or connector, so it tells you the engine works independently of your data access.

In [ ]:
for morsel in session.execute_to_morsels(
    "SELECT name, mass, gravity FROM $planets ORDER BY mass DESC"
):
    print(morsel.to_arrow())